# Download indexed-horizon activation batches

Download the activation files produced by `scripts/run_activation_caching_indexed_horizon_selected_acts.sh` from `indexed_horizon_selected_acts` in GCS. The notebook works in Colab or a local repository checkout, resumes safely by skipping files whose sizes already match GCS, and verifies the completed local copy.

## 1. Install and authenticate

Uncomment the installation command if the notebook environment does not already have the required packages. In Colab, the authentication cell opens the Google sign-in flow; locally it uses Application Default Credentials.

In [ ]:
# %pip install -q google-cloud-storage tqdm python-dotenv

In [ ]:
try:
    from google.colab import auth
except ModuleNotFoundError:
    print('Not running in Colab; using existing Application Default Credentials.')
else:
    auth.authenticate_user()

In [ ]:
import os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path, PurePosixPath

from google.cloud import storage
from tqdm.auto import tqdm

repo_root = Path.cwd()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    pass
else:
    load_dotenv(repo_root / '.env')

## 2. Configure and discover batches

The bucket and indexed-horizon prefix are fixed to the repository's activation-caching locations. The project defaults to the value in `.env.example` and may be overridden through `.env` or an environment variable. Only `activations_batch_*.pt` files directly inside the indexed-horizon prefix are selected.

In [ ]:
PROJECT_ID = os.getenv('GCP_PROJECT_ID', 'temporal-interp-exp')
BUCKET_NAME = 'temporal-research-bucket'
GCS_PREFIX = 'indexed_horizon_selected_acts'
LOCAL_DIR = repo_root / 'data' / 'indexed_horizon_selected_activations'
DOWNLOAD_WORKERS = 8
OVERWRITE = False

if DOWNLOAD_WORKERS < 1:
    raise ValueError('DOWNLOAD_WORKERS must be at least 1.')

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)
prefix_path = PurePosixPath(GCS_PREFIX)
blobs = sorted(
    (
        blob
        for blob in bucket.list_blobs(prefix=f'{GCS_PREFIX}/')
        if PurePosixPath(blob.name).parent == prefix_path
        and PurePosixPath(blob.name).name.startswith('activations_batch_')
        and blob.name.endswith('.pt')
    ),
    key=lambda blob: blob.name,
)
if not blobs:
    raise FileNotFoundError(
        f'No activation batches found at gs://{BUCKET_NAME}/{GCS_PREFIX}/'
    )

total_bytes = sum(int(blob.size or 0) for blob in blobs)
print(f'Remote: gs://{BUCKET_NAME}/{GCS_PREFIX}/')
print(f'Found {len(blobs):,} batches ({total_bytes / 2**30:.2f} GiB).')
print(f'Local destination: {LOCAL_DIR}')

## 3. Download

Files with the expected byte size are skipped unless `OVERWRITE = True`. Missing or incomplete files are downloaded concurrently.

In [ ]:
def download_batch(blob):
    destination = LOCAL_DIR / PurePosixPath(blob.name).name
    expected_size = int(blob.size or 0)
    is_complete = (
        destination.is_file()
        and not OVERWRITE
        and (expected_size == 0 or destination.stat().st_size == expected_size)
    )
    if is_complete:
        status = 'skipped'
    else:
        blob.download_to_filename(str(destination))
        status = 'downloaded'
    return {
        'path': destination,
        'expected_size': expected_size,
        'status': status,
    }


LOCAL_DIR.mkdir(parents=True, exist_ok=True)
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    results = list(
        tqdm(
            executor.map(download_batch, blobs),
            total=len(blobs),
            desc='Downloading indexed-horizon activations',
        )
    )

print(dict(Counter(result['status'] for result in results)))

## 4. Verify the local copy

This checks every object discovered in GCS against its local filename and expected byte size.

In [ ]:
missing = [result['path'] for result in results if not result['path'].is_file()]
size_mismatches = [
    result['path']
    for result in results
    if result['expected_size']
    and result['path'].is_file()
    and result['path'].stat().st_size != result['expected_size']
]
if missing or size_mismatches:
    raise RuntimeError(
        f'Download verification failed: {len(missing)} missing files and '
        f'{len(size_mismatches)} size mismatches.'
    )

verified_bytes = sum(result['path'].stat().st_size for result in results)
print(f'Verified {len(results):,} activation batches ({verified_bytes / 2**30:.2f} GiB).')
print(f'Files are available in: {LOCAL_DIR.resolve()}')